# Dataset — reading and inspecting a raster

The `Dataset` class is pyramids' wrapper around a single raster (here a multi-band GeoTIFF). This tutorial loads a NOAH
daily-precipitation file and walks through the properties every raster carries: its grid dimensions, coordinate
reference system, longitude/latitude axes, bounding box, and geotransform. By the end you will know how to open a raster
and interrogate its spatial metadata.

Every raster you open is a `Dataset` whose properties fall into a few families — grid shape, coordinate
reference system, geographic extent, band/value metadata, and the source it came from:

```mermaid
flowchart TB
    DS(("Dataset<br/>object"))
    DS --> G["<b>grid & shape</b><br/>shape · rows · columns<br/>cell_size · top_left_corner"]
    DS --> C["<b>CRS & coordinates</b><br/>crs · epsg · geotransform<br/>lon · lat · x · y"]
    DS --> E["<b>geographic extent</b><br/>bounds · bbox · total_bounds"]
    DS --> B["<b>bands & values</b><br/>band_count · band_names<br/>no_data_value · dtype"]
    DS --> M["<b>source & metadata</b><br/>raster · file_name<br/>driver_type · meta_data"]
    DS --> N["<b>NetCDF subclass adds</b><br/>variables · time_stamp"]
```

The rest of this tutorial reads these one group at a time.

## Setup

Import the `Dataset` class and point at the sample raster. The path is written relative to this notebook so the example
runs from a fresh checkout. Reading the file emits a one-time log line as pyramids configures GDAL.

In [ ]:
# NBVAL_IGNORE_OUTPUT
from pyramids.dataset import Dataset

%matplotlib inline

path = r"../../../examples/data/geotiff/noah-precipitation-1979.tif"

## Read any raster format

`Dataset.read_file` auto-detects the driver from the file and returns a `Dataset`. The same call works for GeoTIFF,
NetCDF, and any other format GDAL can open — you don't pick a reader per format.

In [ ]:
dataset = Dataset.read_file(path)

## Explore the dataset

Printing a `Dataset` gives a compact summary of its geo-metadata: corner coordinates, cell size, grid shape, CRS, band
list, no-data mask, and dtype. It's the quickest way to sanity-check a file after loading it.

In [ ]:
print(dataset)

The summary confirms a global 0.5-degree grid (360 rows x 720 columns), 4 bands, EPSG:4326, and a `float32` no-data mask
around -9.97e+36. The four bands are the first four daily precipitation fields of 1979.

## Plot a band

`Dataset.plot` renders one band through cleopatra and returns an `ArrayGlyph` (its matplotlib handles live on `.fig` /
`.ax` if you need to tweak them). The key options:

| parameter | meaning | typical value |
| --- | --- | --- |
| `band` | which band to draw (0-based) | `0` |
| `vmax` | upper clip for the colour scale | `30` (mm/day) |
| `cbar_label` | colour-bar caption | `"Rainfall mm/day"` |
| `cbar_length` | colour-bar length as a fraction of the axis | `0.85` |
| `figsize` | figure size in inches | `(10, 5)` |

In [ ]:
array_glyph = dataset.plot(
    band=0,
    figsize=(10, 5),
    title="Noah daily Precipitation 1979-01-01",
    cbar_label="Rainfall mm/day",
    vmax=30,
    cbar_length=0.85,
)

The map shows the first day's precipitation on a global grid. Because this file uses a 0-360 longitude axis, the layout
is centered on the antimeridian — see the companion `convert-longitude` notebook for the fix.

## Grid dimensions

A raster is a cube of `(bands, rows, columns)`. These properties expose each axis separately so you can size arrays or
loop over bands without touching GDAL directly.

In [ ]:
print(f"Dataset dimensions: {dataset.shape}")
print(f"Dataset rows: {dataset.rows}")
print(f"Dataset columns: {dataset.columns}")
print(f"Dataset number of bands: {dataset.band_count}")

`shape` is `(4, 360, 720)` — 4 bands over a 360x720 grid — and `band_count` is 4, matching the four daily fields seen in
the summary.

### Cell size

The cell size is the ground resolution of one pixel, in the CRS units (degrees here). 0.5 means each pixel is half a
degree on a side.

In [ ]:
print(f"Cell size: {dataset.cell_size}")

### Band names

Each band carries a name. NOAH daily fields import with generic `Band_1 ... Band_4` labels, which you can use to select
or relabel bands.

In [ ]:
dataset.band_names

## Spatial reference

The CRS ties the pixel grid to real-world coordinates. `epsg` is the numeric code (4326 = WGS84 lon/lat) and `crs` is
the full WKT definition.

In [ ]:
print(f"EPSG: {dataset.epsg}")
print(f"Coordinate reference system: {dataset.crs}")

EPSG 4326 is geographic WGS84, so coordinates are in degrees of longitude/latitude. The next two properties give the
coordinate of each column (`lon`) and each row (`lat`).

In [ ]:
dataset.lon

`lat` is the matching per-row latitude axis. It runs from +89.75 down to -89.75 because raster rows are ordered
north-to-south (top-left origin).

In [ ]:
dataset.lat

## Bounding box and bounds

`bbox` returns the extent as a plain `[xmin, ymin, xmax, ymax]` list, while `bounds` returns it as a GeoDataFrame polygon
you can plot or intersect with other vector layers.

In [ ]:
dataset.bbox

`bounds` wraps the same extent as a one-row GeoDataFrame, so the footprint can be treated as vector geometry.

In [ ]:
print(dataset.bounds)

Because `bounds` is a GeoDataFrame, its `.plot()` draws the raster footprint as a rectangle.

In [ ]:
dataset.bounds.plot()

The rectangle is the raster's ground footprint — here the full globe (0..360 lon, -90..90 lat) in the file's native
coordinates.

### Geotransform

The GDAL geotransform is the six-number affine mapping from pixel (col, row) to world (x, y):
`(x_origin, pixel_width, 0, y_origin, 0, pixel_height)`. A negative pixel height means rows run north-to-south.

In [ ]:
dataset.geotransform

### Top-left corner

`top_left_corner` is the world coordinate of the grid's origin — surfaced as a convenient tuple from the geotransform.

In [ ]:
dataset.top_left_corner